In [1]:
import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "sqlalchemy"], check=True)
print("Dependencies installed!")

Dependencies installed!


# 02 — Customers Transformation
Reads raw.customers from PostgreSQL, validates and cleans the data using Pandas,
then loads clean records into staging.customers.

In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timezone

engine = create_engine(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

print("Connected!")

Connected!


## Step 1 — Load raw customers

In [4]:
df = pd.read_sql("SELECT * FROM raw.customers;", engine)

print(f"Loaded {len(df):,} raw customer records")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
df.head()

Loaded 282 raw customer records

Columns: ['customer_id', 'name', 'email', 'phone', 'city', 'batch_id', 'created_at']

Data types:
customer_id            object
name                   object
email                  object
phone                  object
city                   object
batch_id                int64
created_at     datetime64[ns]
dtype: object


,customer_id,name,email,phone,city,batch_id,created_at
0,cust57705,Jack Owens,thernandez@example.net,877.553.8097,Beasleyborough,1,2026-04-18 03:26:28.322
1,cust27497,Eric Clark,jason94@example.org,808.482.0948,Howardberg,1,2026-04-18 03:26:28.317
2,cust67400,Mark Ballard,huntlaurie@example.org,5662555334,West Kelly,1,2026-04-18 03:26:28.319
3,cust24724,John Conley,wilkersonfelicia@example.org,(955)933-0606x9343,Dwaynefort,1,2026-04-18 03:26:28.318
4,cust83487,Steven Andrews,ubryant@example.com,001-400-260-4088,Ashleyborough,1,2026-04-18 03:26:28.321


## Step 2 — Validate

In [7]:
# Check for nulls in required fields
required_fields = ["customer_id", "name", "email"]

print("NULL CHECKS")
for field in required_fields:
    null_count = df[field].isnull().sum()
    status = "PASS" if null_count == 0 else "FAIL"
    print(f"  [{status}] {field}: {null_count} nulls")

# Check for duplicates on customer_id
duplicates = df["customer_id"].duplicated().sum()
print(f"\n DUPLICATE CHECK ")
print(f"  [{'PASS' if duplicates == 0 else 'FAIL'}] customer_id duplicates: {duplicates}")

# Check created_at nulls — known issue from early batches
created_at_nulls = df["created_at"].isnull().sum()
print(f"\n KNOWN ISSUE ")
print(f"  created_at nulls: {created_at_nulls} (will be filled with sentinel 1970-01-01)")

NULL CHECKS
  [PASS] customer_id: 0 nulls
  [PASS] name: 0 nulls
  [PASS] email: 0 nulls

 DUPLICATE CHECK 
  [PASS] customer_id duplicates: 0

 KNOWN ISSUE 
  created_at nulls: 0 (will be filled with sentinel 1970-01-01)


## Step 3 — Clean and Transform

In [15]:
df_clean = df.copy()

# Normalize email to lowercase
df_clean["email"] = df_clean["email"].str.strip().str.lower()

# Strip whitespace from text fields
df_clean["name"] = df_clean["name"].str.strip()
df_clean["city"] = df_clean["city"].str.strip()

# Normalize name to title case — handles ALL CAPS or all lowercase edge cases
# "JOHN CONLEY" → "John Conley"
# "john conley" → "John Conley"
df_clean["name"] = df_clean["name"].str.strip().str.title()

# Normalize phone — remove spaces, dashes and dots
# Keeps digits and + for international format compatibility
df_clean["phone"] = df_clean["phone"].str.replace(r"[\s\-\.]", "", regex=True)

# Fill null created_at with sentinel 1970-01-01
SENTINEL = pd.Timestamp("1970-01-01", tz="UTC")
df_clean["created_at"] = df_clean["created_at"].fillna(SENTINEL)

# Add loaded_at timestamp
df_clean["loaded_at"] = datetime.now(timezone.utc)

print(f"Cleaned {len(df_clean):,} customer records")
print(df_clean["phone"].head(5).to_string())
df_clean.head()

Cleaned 282 customer records
0           8775538097
1           8084820948
2           5662555334
3    (955)9330606x9343
4        0014002604088


,customer_id,name,email,phone,city,batch_id,created_at,loaded_at
0,cust57705,Jack Owens,thernandez@example.net,8775538097,Beasleyborough,1,2026-04-18 03:26:28.322,2026-04-22 16:40:17.784846+00:00
1,cust27497,Eric Clark,jason94@example.org,8084820948,Howardberg,1,2026-04-18 03:26:28.317,2026-04-22 16:40:17.784846+00:00
2,cust67400,Mark Ballard,huntlaurie@example.org,5662555334,West Kelly,1,2026-04-18 03:26:28.319,2026-04-22 16:40:17.784846+00:00
3,cust24724,John Conley,wilkersonfelicia@example.org,(955)9330606x9343,Dwaynefort,1,2026-04-18 03:26:28.318,2026-04-22 16:40:17.784846+00:00
4,cust83487,Steven Andrews,ubryant@example.com,0014002604088,Ashleyborough,1,2026-04-18 03:26:28.321,2026-04-22 16:40:17.784846+00:00


## Step 4 — Load to staging.customers

In [12]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("""
        CREATE SCHEMA IF NOT EXISTS staging;
    """))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS staging.customers (
            customer_id     VARCHAR PRIMARY KEY,
            name            VARCHAR NOT NULL,
            email           VARCHAR NOT NULL,
            phone           VARCHAR,
            city            VARCHAR,
            batch_id        INTEGER,
            created_at      TIMESTAMP WITH TIME ZONE,
            loaded_at       TIMESTAMP WITH TIME ZONE
        );
    """))
    conn.commit()

print("Staging schema and customers table ready!")

Staging schema and customers table ready!


In [13]:
# Write clean records to staging.customers
# if_exists="append" adds rows without dropping the table
# index=False prevents pandas from writing the DataFrame index as a column
df_clean.to_sql(
    name="customers",
    schema="staging",
    con=engine,
    if_exists="append",
    index=False,
)

print(f"Loaded {len(df_clean):,} customers into staging.customers")

Loaded 282 customers into staging.customers


## Step 5 — Verify

In [14]:
df_staging = pd.read_sql("SELECT * FROM staging.customers;", engine)

print(f"staging.customers row count: {len(df_staging):,}")
print(f"\nNull counts:\n{df_staging.isnull().sum()}")
df_staging.head()

staging.customers row count: 282

Null counts:
customer_id    0
name           0
email          0
phone          0
city           0
batch_id       0
created_at     0
loaded_at      0
dtype: int64


,customer_id,name,email,phone,city,batch_id,created_at,loaded_at
0,cust57705,Jack Owens,thernandez@example.net,8775538097,Beasleyborough,1,2026-04-18 03:26:28.322000+00:00,2026-04-22 16:30:26.968878+00:00
1,cust27497,Eric Clark,jason94@example.org,8084820948,Howardberg,1,2026-04-18 03:26:28.317000+00:00,2026-04-22 16:30:26.968878+00:00
2,cust67400,Mark Ballard,huntlaurie@example.org,5662555334,West Kelly,1,2026-04-18 03:26:28.319000+00:00,2026-04-22 16:30:26.968878+00:00
3,cust24724,John Conley,wilkersonfelicia@example.org,(955)9330606x9343,Dwaynefort,1,2026-04-18 03:26:28.318000+00:00,2026-04-22 16:30:26.968878+00:00
4,cust83487,Steven Andrews,ubryant@example.com,0014002604088,Ashleyborough,1,2026-04-18 03:26:28.321000+00:00,2026-04-22 16:30:26.968878+00:00
